01_extraction_gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/01_extraction_gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/01_extraction_gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Extraction with Gemini: Structuring Maintenance Logs

## What you will learn
- Use an LLM to extract structured fields from unstructured text.
- Verify extracted data against a known ground truth.
- Identify extraction errors: missed fields, fabricated data, partial extractions.

**NLP Task**: Extraction — pulling specific structured information out of unstructured text.

**Scenario**: NordWind Energy technicians write free-text maintenance logs.
We need to extract: turbine_id, component, issue_type, severity, and date.

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai pandas


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Sample Data: NordWind Energy Maintenance Logs

These are realistic free-text maintenance logs written by field technicians.
Each log has a known ground truth that we'll compare against.


In [ ]:
# Maintenance logs with ground truth for verification
MAINTENANCE_LOGS = [
    {
        "log_text": "Visited Turbine NW-107 at Ringkøbing site on 14 March 2026. Gearbox showing excessive vibration at high wind speeds (>18 m/s). Replaced bearing assembly in main gearbox. Oil sample taken — results pending. Technician: Lars Henriksen.",
        "ground_truth": {
            "turbine_id": "NW-107",
            "component": "gearbox",
            "issue_type": "excessive vibration",
            "severity": "high",
            "date": "2026-03-14"
        }
    },
    {
        "log_text": "Routine inspection NW-042, Thyborøn wind farm, 11 Mar. Blade pitch system responding normally. Minor corrosion on nacelle access ladder — flagged for next scheduled maintenance. No immediate action. — M. Sørensen",
        "ground_truth": {
            "turbine_id": "NW-042",
            "component": "nacelle access ladder",
            "issue_type": "minor corrosion",
            "severity": "low",
            "date": "2026-03-11"
        }
    },
    {
        "log_text": "URGENT: NW-089 Hvide Sande — yaw motor failure during storm conditions 12/03/2026. Turbine stuck facing 47° off wind direction. Emergency shutdown initiated. Yaw drive motor #2 burned out. Replacement unit ordered, ETA 3 days. Site unsafe for climbing until wind drops below 15 m/s. Contact: K. Vestergaard",
        "ground_truth": {
            "turbine_id": "NW-089",
            "component": "yaw motor",
            "issue_type": "motor failure",
            "severity": "critical",
            "date": "2026-03-12"
        }
    },
    {
        "log_text": "NW-215 at Esbjerg offshore platform. Performed quarterly transformer oil test on 10 March 2026. Dielectric strength within acceptable range (62 kV). Noted slight discolouration in oil sample — recommend follow-up in 6 weeks rather than standard 12-week interval. Signed off by P. Nielsen, Level 3 cert.",
        "ground_truth": {
            "turbine_id": "NW-215",
            "component": "transformer",
            "issue_type": "oil discolouration",
            "severity": "medium",
            "date": "2026-03-10"
        }
    },
    {
        "log_text": "Checked turbines 301 through 305 at Lemvig cluster, 13 March. All SCADA readings normal. NW-303 had a lightning strike mark on blade 2 tip — surface damage only, no structural concern per visual inspection. Will schedule drone survey to confirm. —A. Christensen",
        "ground_truth": {
            "turbine_id": "NW-303",
            "component": "blade",
            "issue_type": "lightning strike damage",
            "severity": "medium",
            "date": "2026-03-13"
        }
    },
    {
        "log_text": "13 Mar 2026 — NW-156, Hanstholm. Generator winding temperature alarm triggered at 14:22. On-site inspection found cooling fan belt loose. Tightened and monitored for 2 hours — temps returned to normal operating range (85-92°C). Root cause: belt tensioner bracket fatigue. Replaced bracket. Tech: J. Møller",
        "ground_truth": {
            "turbine_id": "NW-156",
            "component": "generator cooling fan",
            "issue_type": "temperature alarm",
            "severity": "high",
            "date": "2026-03-13"
        }
    },
    {
        "log_text": "Annual safety inspection completed for NW-078 Nymindegab on 9 March. Fire suppression system tested OK. Emergency lighting functional. Fall arrest anchors recertified. One issue: anemometer on nacelle reading 12% high compared to met mast — recalibrated. All docs updated in SAP.",
        "ground_truth": {
            "turbine_id": "NW-078",
            "component": "anemometer",
            "issue_type": "sensor calibration error",
            "severity": "low",
            "date": "2026-03-09"
        }
    },
    {
        "log_text": "Called out to NW-412 after SCADA showed pitch angle stuck at 3.2° for the whole day (should vary with wind). Turned out the pitch encoder cable had a rodent chew through. Replaced cable and encoder. Also found bird nesting material in the hub — cleaned out. 11 March, Holmsland site. B. Andersen.",
        "ground_truth": {
            "turbine_id": "NW-412",
            "component": "pitch encoder",
            "issue_type": "cable damage (rodent)",
            "severity": "high",
            "date": "2026-03-11"
        }
    },
]

logs_df = pd.DataFrame(MAINTENANCE_LOGS)
print(f'Loaded {len(logs_df)} maintenance logs.')
logs_df[['log_text']].head(3)


## Extract Structured Fields with Gemini

We ask Gemini to pull out specific fields from each free-text log.
The key evaluation question: Did it find everything? Did it fabricate anything?


In [ ]:
EXTRACTION_PROMPT = """Extract the following fields from this wind turbine maintenance log.
Return ONLY valid JSON with these exact keys:
- turbine_id (string, format NW-XXX)
- component (string, the main component affected)
- issue_type (string, brief description of the problem)
- severity (string, one of: low, medium, high, critical)
- date (string, format YYYY-MM-DD)

If a field cannot be determined from the text, use null.

Maintenance log:
\"\"\"{log_text}\"\"\"

JSON output:"""

def extract_fields(log_text: str) -> dict:
    prompt = EXTRACTION_PROMPT.format(log_text=log_text)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        raw = resp.text.strip()
        # Clean markdown code fences if present
        if raw.startswith('```'):
            raw = raw.split('\n', 1)[1] if '\n' in raw else raw[3:]
        if raw.endswith('```'):
            raw = raw.rsplit('```', 1)[0]
        raw = raw.strip()
        data = json.loads(raw)
        return {'extracted': data, 'error': None}
    except json.JSONDecodeError as e:
        return {'extracted': {}, 'error': f'JSON parse error: {e}. Raw: {raw[:200]}'}
    except Exception as e:
        return {'extracted': {}, 'error': str(e)}

# Run extraction
results = []
for idx, row in logs_df.iterrows():
    result = extract_fields(row['log_text'])
    results.append(result)
    time.sleep(0.5)

logs_df['extracted'] = [r['extracted'] for r in results]
logs_df['error'] = [r['error'] for r in results]

print('Extraction complete.')


## Compare Extracted vs. Ground Truth

We check each field individually to understand *which types of information*
the model handles well and where it struggles.


In [ ]:
FIELDS = ['turbine_id', 'component', 'issue_type', 'severity', 'date']

comparison_rows = []
for idx, row in logs_df.iterrows():
    gt = row['ground_truth']
    ex = row['extracted'] if isinstance(row['extracted'], dict) else {}

    for field in FIELDS:
        gt_val = gt.get(field, '')
        ex_val = ex.get(field, '')

        # Normalize for comparison
        gt_norm = str(gt_val).lower().strip() if gt_val else ''
        ex_norm = str(ex_val).lower().strip() if ex_val else ''

        # Fuzzy match: check if ground truth is contained in extraction or vice versa
        exact_match = gt_norm == ex_norm
        fuzzy_match = (gt_norm in ex_norm) or (ex_norm in gt_norm) if gt_norm and ex_norm else False

        comparison_rows.append({
            'log_index': idx,
            'turbine': gt.get('turbine_id', ''),
            'field': field,
            'ground_truth': gt_val,
            'extracted': ex_val,
            'exact_match': exact_match,
            'fuzzy_match': fuzzy_match,
            'fabricated': bool(ex_val) and not gt_val,
            'missed': bool(gt_val) and not ex_val,
        })

comp_df = pd.DataFrame(comparison_rows)

# Summary by field
print('=== Extraction Accuracy by Field ===')
field_summary = comp_df.groupby('field').agg(
    exact_match_rate=('exact_match', 'mean'),
    fuzzy_match_rate=('fuzzy_match', 'mean'),
    missed_count=('missed', 'sum'),
).round(2)
print(field_summary)
print()

# Show specific mismatches
mismatches = comp_df[~comp_df['exact_match'] & ~comp_df['fuzzy_match']]
print(f'=== Mismatches ({len(mismatches)}) ===')
for _, row in mismatches.iterrows():
    print(f'  Log {row["log_index"]} ({row["turbine"]}) — {row["field"]}')
    print(f'    Expected: {row["ground_truth"]}')
    print(f'    Got:      {row["extracted"]}')
    print()


## Discussion Questions

1. **Which fields are easiest/hardest to extract?**
   - turbine_id and date are structured → easier
   - severity and issue_type require interpretation → harder

2. **What happens when the model fabricates data?**
   - "Fabrication" in extraction means adding info not in the source
   - This is a hallucination specific to extraction tasks

3. **How would NordWind use this in practice?**
   - Auto-populate maintenance database from technician notes
   - But what's the cost of a wrong turbine_id vs. wrong severity?

## Export for Student Review


In [ ]:
export_df = comp_df[['log_index', 'turbine', 'field', 'ground_truth', 'extracted', 'exact_match', 'fuzzy_match']]
export_df = export_df.copy()
export_df['student_notes'] = ''
export_path = 'extraction_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
